# Global vs. decentralized POMDP policy: improvement over swap-asap

For a grid of `(p, cutoff)` values, this notebook computes the exact expected delivery time of three policies over the same discrete-event MDP (`environment.py` / `policy.py`):

- **swap-asap** — the baseline: always perform every valid swap (`policy_eval_swapasap`).
- **global-knowledge policy** — the optimal policy from full-chain policy iteration (`policy_iteration`): a central controller sees every qubit's age/hanging status every time slot.
- **decentralized POMDP policy** — `po_policy.py`'s self-consistent, belief-space policy iteration (`pomdp_policy_iteration`): each node sees only its own one or two qubits, and instead conditions its action on a Bayesian belief over the full chain state rather than on the bare local state (`distill.py`) or the full state (`policy.py`). See `po_policy.py`'s module docstring for the planning algorithm and its approximations.

Both are evaluated exactly (no Monte Carlo noise), so the two ratios below are directly comparable.

It then plots two heatmaps of `(T_swap-asap - T_policy) / T_policy` — the fractional improvement over swap-asap — for the global and POMDP policies, on `p` (x) vs. `cutoff` (y).

**Tractability note:** unlike the global and swap-asap solves, `pomdp_policy_iteration` plans over an augmented hyper-state (true state × every node's belief), which is far more expensive and, for some `(p, cutoff)` points, can exceed `MAX_HYPERSTATES` before converging (see `po_policy.py`'s docstring on belief rounding). Grid points that hit this cap are left blank (shown in light gray, annotated `n/a`) rather than failing the whole notebook — widening `PROB_FLOOR`/lowering `BELIEF_PRECISION` trades some accuracy for headroom on those points.

All the underlying data is generated by this notebook (no external scripts to run first): each result is cached to disk under `data_policyiter/`, `data_swapasap/`, and `data_pomdppolicy/` the first time it's computed, exactly like running `policy.py` / `po_policy.py` from the command line, so re-running the notebook later only recomputes what's missing.

In [1]:
import sys
from pathlib import Path

# Locate src/ (where environment.py / policy.py / po_policy.py live) regardless
# of whether Jupyter's cwd is src/ itself or the repo root.
_candidates = [Path.cwd(), Path.cwd() / "src"]
SRC_DIR = next((c for c in _candidates if (c / "environment.py").exists()), None)
if SRC_DIR is None:
    raise RuntimeError(
        "Couldn't find environment.py -- run this notebook from the repo's "
        "src/ directory, or from the repo root."
    )
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from policy import (
    check_policyiter_data,
    load_policyiter_data,
    policy_iteration,
    check_swapasap_data,
    load_swapasap_data,
    policy_eval_swapasap,
)
from po_policy import (
    check_pomdppolicy_data,
    load_pomdppolicy_data,
    pomdp_policy_iteration,
)

## Parameters

Adjust the chain length, swap probability, and grid resolution here.

`N` is kept small (3) relative to `local_policy_heatmap.ipynb`'s `N=5`: the POMDP solve plans over (true state × every node's belief), which grows far faster than the fully-observed MDP `policy.py` solves, so a 3-node chain already exercises real belief tracking (the middle node's two qubits vs. the two end nodes' one qubit each) while staying tractable across a full grid. `BELIEF_PRECISION`/`PROB_FLOOR` control the belief-rounding approximation `po_policy.py` uses to keep its hyper-state space finite (see that module's docstring) -- coarser values (fewer decimals, higher floor) trade some accuracy for a much smaller state space.

In [2]:
N = 4                     # number of nodes in the chain
P_S = 1.0                 # entanglement swap success probability
TOLERANCE = 1e-5          # policy-iteration value tolerance (both policy.py and po_policy.py)
ALLOW_DISCARD = False     # whether qubit discard is enabled (po_policy.py only supports
                          # False or True -- see po_policy.py's _check_pomdp_discard_support)

BELIEF_PRECISION = 1      # decimal places each node's belief is rounded to before being
                          # treated as a distinct belief-state -- bounds the otherwise-
                          # unbounded belief space (see po_policy.py's module docstring)
PROB_FLOOR = 0.2          # belief-support entries at or below this probability are pruned
                          # (renormalizing what remains) before keying on the belief
MAX_HYPERSTATES = 20000   # safety cap on the POMDP hyper-state graph; grid points that
                          # exceed it are recorded as NaN rather than aborting the notebook

P_VALUES = np.round(np.arange(0.3, 0.91, 0.1), 2)   # x-axis: link generation probability
CUTOFF_VALUES = list(range(2, 5))                     # y-axis: qubit-age cutoff

print(f"n={N}, p_s={P_S}, discard={'on' if ALLOW_DISCARD else 'off'}, "
      f"belief_precision={BELIEF_PRECISION}, prob_floor={PROB_FLOOR}")
print(f"p in {list(P_VALUES)}")
print(f"cutoff in {CUTOFF_VALUES}")
print(f"grid size: {len(CUTOFF_VALUES) * len(P_VALUES)} points")

n=4, p_s=1.0, discard=off, belief_precision=1, prob_floor=0.2
p in [np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9)]
cutoff in [2, 3, 4]
grid size: 21 points


## Generate the data

Each helper ensures its underlying policy exists (computing and caching it if not) and returns the exact expected delivery time `-(value + 1)` (see `policy.py`'s `Agent` docstring for the sign convention). `pomdp_delivery_time` additionally catches the `RuntimeError` `pomdp_policy_iteration` raises when a grid point's hyper-state graph exceeds `MAX_HYPERSTATES`, returning `NaN` for that point instead of aborting the whole sweep.

In [3]:
def swapasap_delivery_time(n, p, p_s, cutoff, tol, allow_discard):
    """Exact expected delivery time under the swap-asap baseline policy."""
    if not check_swapasap_data(n, p, p_s, cutoff, tol, allow_discard):
        policy_eval_swapasap(n, p, p_s, cutoff, tolerance=tol, progress=False,
                              savedata=True, allow_discard=allow_discard)
    _, state_info, _ = load_swapasap_data(n, p, p_s, cutoff, tol, allow_discard)
    return -(state_info[0]["value"] + 1)


def optimal_delivery_time(n, p, p_s, cutoff, tol, allow_discard):
    """Exact expected delivery time under the global-knowledge optimal policy."""
    if not check_policyiter_data(n, p, p_s, cutoff, tol, allow_discard):
        policy_iteration(n, p, p_s, cutoff, tolerance=tol, progress=False,
                          savedata=True, allow_discard=allow_discard)
    _, state_info, _ = load_policyiter_data(n, p, p_s, cutoff, tol, allow_discard)
    return -(state_info[0]["value"] + 1)


def pomdp_delivery_time(n, p, p_s, cutoff, tol, allow_discard, belief_precision, prob_floor,
                         max_hyperstates):
    """Exact expected delivery time under the self-consistent, decentralized POMDP
    policy (distills/solves it first if it hasn't been already). Returns NaN, with a
    printed warning, if this grid point's hyper-state graph exceeds max_hyperstates
    before converging -- see po_policy.py's module docstring for why the belief space
    isn't generally finite without the belief_precision/prob_floor approximation."""
    if check_pomdppolicy_data(n, p, p_s, cutoff, tol, belief_precision, prob_floor, allow_discard):
        v0_evol, _node_policy, _exe_time = load_pomdppolicy_data(
            n, p, p_s, cutoff, tol, belief_precision, prob_floor, allow_discard)
        return -(v0_evol[-1][-1] + 1)
    try:
        v0_evol, _node_policy, _exe_time = pomdp_policy_iteration(
            n, p, p_s, cutoff, tolerance=tol, belief_precision=belief_precision,
            prob_floor=prob_floor, max_hyperstates=max_hyperstates, progress=False,
            savedata=True, allow_discard=allow_discard)
        return -(v0_evol[-1][-1] + 1)
    except RuntimeError as exc:
        print(f"  [pomdp] p={p:.1f} cutoff={cutoff}: {exc}")
        return np.nan

In [4]:
shape = (len(CUTOFF_VALUES), len(P_VALUES))
T_swapasap = np.empty(shape)
T_optimal = np.empty(shape)
T_pomdp = np.empty(shape)

for i, cutoff in enumerate(CUTOFF_VALUES):
    for j, p in enumerate(P_VALUES):
        T_swapasap[i, j] = swapasap_delivery_time(N, p, P_S, cutoff, TOLERANCE, ALLOW_DISCARD)
        T_optimal[i, j] = optimal_delivery_time(N, p, P_S, cutoff, TOLERANCE, ALLOW_DISCARD)
        T_pomdp[i, j] = pomdp_delivery_time(N, p, P_S, cutoff, TOLERANCE, ALLOW_DISCARD,
                                             BELIEF_PRECISION, PROB_FLOOR, MAX_HYPERSTATES)
        pomdp_str = f"{T_pomdp[i, j]:.3f}" if not np.isnan(T_pomdp[i, j]) else "n/a"
        print(f"cutoff={cutoff}  p={p:.1f}   "
              f"T_swap-asap={T_swapasap[i, j]:.3f}   "
              f"T_optimal={T_optimal[i, j]:.3f}   "
              f"T_pomdp={pomdp_str}")

KeyboardInterrupt: 

In [ ]:
# Fractional improvement over swap-asap: positive means the policy delivers
# faster than swap-asap, negative means swap-asap actually wins. NaN (grid
# points where the POMDP hyper-state graph exceeded MAX_HYPERSTATES) propagates
# through untouched, so those cells stay blank in the heatmap below.
ratio_global = (T_swapasap - T_optimal) / T_optimal
ratio_pomdp = (T_swapasap - T_pomdp) / T_pomdp

## Visualize

Both heatmaps share one diverging color scale (blue = beats swap-asap, red = loses to it, gray = ties) so the panels are directly comparable. Grid points where the POMDP solve hit `MAX_HYPERSTATES` (see the printed warnings above) render as light gray with an `n/a` label instead of a colored value.

In [ ]:
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
NAN_FILL = "#dedad2"

# Diverging blue <-> red pair with a neutral gray midpoint at zero.
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "blue_gray_red", ["#e34948", "#f0efec", "#2a78d6"]
)
DIVERGING_CMAP.set_bad(NAN_FILL)


def _text_color_for(rgba):
    """Ink or white, whichever contrasts with a cell's fill."""
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    return INK_PRIMARY if luminance > 0.6 else "#ffffff"


def draw_heatmap(ax, matrix, norm, title):
    masked = np.ma.masked_invalid(matrix)
    im = ax.imshow(masked, cmap=DIVERGING_CMAP, norm=norm, aspect="auto", origin="lower")

    ax.set_xticks(range(len(P_VALUES)))
    ax.set_xticklabels([f"{p:.1f}" for p in P_VALUES], color=INK_MUTED)
    ax.set_yticks(range(len(CUTOFF_VALUES)))
    ax.set_yticklabels(CUTOFF_VALUES, color=INK_MUTED)
    ax.set_xlabel("p  (link generation success probability)", color=INK_SECONDARY)
    ax.set_ylabel("cutoff", color=INK_SECONDARY)
    ax.set_title(title, color=INK_PRIMARY, fontsize=11.5, fontweight="semibold", pad=8)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

    # A thin surface-color gap between cells, so neighbors read as distinct
    # without drawing a border around every cell.
    ax.set_xticks(np.arange(-0.5, len(P_VALUES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(CUTOFF_VALUES), 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]
            if np.isnan(value):
                ax.text(col, row, "n/a", ha="center", va="center",
                         color=INK_MUTED, fontsize=8)
                continue
            text_color = _text_color_for(im.cmap(im.norm(value)))
            ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                     color=text_color, fontsize=8)

    return im


combined = np.concatenate([ratio_global.ravel(), ratio_pomdp.ravel()])
vmax = float(np.nanmax(np.abs(combined)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor=SURFACE)
for ax, matrix, title in zip(
    axes,
    [ratio_global, ratio_pomdp],
    ["Global-knowledge policy", "Decentralized POMDP policy"],
):
    ax.set_facecolor(SURFACE)
    im = draw_heatmap(ax, matrix, norm, title)

fig.subplots_adjust(top=0.80, wspace=0.35)

cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label("(T_swap-asap − T_policy) / T_policy", color=INK_SECONDARY)
cbar.ax.yaxis.set_tick_params(color=INK_MUTED, labelcolor=INK_MUTED)
cbar.outline.set_visible(False)

fig.suptitle(
    f"Improvement over swap-asap  (n={N} nodes, p_s={P_S}, discard={'on' if ALLOW_DISCARD else 'off'})",
    color=INK_PRIMARY, fontsize=14, fontweight="semibold", y=0.98,
)
plt.show()

With the defaults above (`n=3`, `p_s=0.8`), both panels are almost uniformly a hair below zero (on the order of `1e-5`, i.e. a few hundredths of a percent) -- the global-knowledge policy and the decentralized POMDP policy both essentially *match* swap-asap rather than clearly beating it. That's an expected finding here, not a sign anything's broken: a 3-node chain only ever has one swap decision to make (the single middle node), so there's very little room for *when* to swap to matter -- the gains `policy_iteration`'s own optimal-cutoff literature reports grow with chain length, and `n=3` is close to the smallest case where a swap decision exists at all. Reassuringly, the POMDP panel tracks the global panel almost exactly everywhere it converged -- belief tracking is recovering essentially all of the (small) available gain here, which is itself a useful sanity check on `po_policy.py`'s solve, even though the effect size is tiny.

Things worth trying from here, to get a panel with more visible structure:
- **Lower `P_S`** (the swap success probability) -- a less reliable swap raises the value of waiting for a better-aged partner before attempting one, which is exactly where swap-asap starts to lose ground (see `local_policy_heatmap.ipynb`'s own `p_s=1.0`, `n=5` grid for a much more dramatic version of this effect).
- **Raise `N`** -- more middle nodes means more swap decisions and more room for timing to matter, but the POMDP hyper-state space grows much faster than the fully-observed one; expect to need a coarser `BELIEF_PRECISION`/larger `PROB_FLOOR` (and a larger `MAX_HYPERSTATES`, and more patience) to keep the grid tractable at `n=4` or beyond.
- **Compare against `distill.py`'s distilled local policy** at the same parameters (see `local_policy_heatmap.ipynb`) -- since both it and the POMDP policy are decentralized, the gap between them isolates how much belief-tracking specifically buys you over a policy that only ever sees the bare current local state.